In [28]:
import torch
import sys
sys.path.append("../model")
from gpt import GPT_124,GPT_CONFIG_124M
import tiktoken

In [29]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer=tiktoken.get_encoding("gpt2")

In [30]:
checkpoint = torch.load("gpt124m.pt", map_location=device)
model = GPT_124(checkpoint["cfg"]).to(device)
model.load_state_dict(checkpoint["model_state_dict"])

<All keys matched successfully>

Adding temperature on early version

In [31]:
def text_to_token_ids(text, tokenizer): 
    """ Takes input text converts to tokens and make em ready to pass to gpt"""
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # add batch dimension
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0) # remove batch dimension
    return tokenizer.decode(flat.tolist())


In [ ]:
def text_gen(model,ids,max_token,context_size,temp): ## Needed for inference
    """ 
    ids is input tokens (batch,n_tokens)
    Take ids and gens next max_token ids
    
    What new we did 
    logits-> logits/temp-> softmax-> sample from multinomial Distribution
    """
    model.eval()
    for i in range(max_token):
        ids_cl=ids[:,-context_size:] 
        with torch.no_grad():
            out=model(ids_cl)
        last_logit=out[:,-1,:] 
        if temp > 0:
            last_logit=last_logit/temp
            probs=torch.softmax(last_logit,dim=-1)
            token=torch.multinomial(probs,num_samples=1)
            ids=torch.cat((ids,token),dim=1)
        else:
            probs=torch.softmax(last_logit,dim=-1) 
            token=torch.argmax(probs,keepdim=True,dim=-1)
            ids=torch.cat((ids,token),dim=1) 
    return ids    

In [33]:
def generate_text(model, tokenizer, device, start_context,max_token,temp):
    """
    Takes starting context->token_id-> generate token(max_tokens)-> decode ->print output
        
    """
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device) ## encode a starting sample

    with torch.no_grad():
        token_ids = text_gen(
            model=model, ids=encoded,
            max_token=max_token, context_size=context_size,temp=temp
        ) ## get token id of generated text upto max

    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " "))  # Compact print format
    model.train()

In [35]:
token_ids = generate_text(
 model=model,
 tokenizer=tokenizer,start_context="Every effort moves you fast",
 max_token=25,device=device,temp=1
)

Every effort moves you fast, heicket such that mighty wantushiCalling one of the ease--it. Poor one not to him--rather?" 
